## 0 · The Challenge

> **Three months after the hallucination guard went live.** Palermo International's technical
> contact sends a follow-up email:
>
> _"Your hallucination guard is working well — we haven't seen another Marchetti incident.
> But I notice the system now attaches a confidence percentage to some answers:
> '87% confident.' Our editors are treating this like a probability. Does 87% confident
> actually mean the model is right 87% of the time? Because if not, that number is
> worse than useless — it gives our people false certainty."_

This is the **calibration problem**. A _well-calibrated_ model that says "80% confident"
should be correct 80% of the time — not 95% (overconfident) or 60% (underconfident).

Most LLMs are severely miscalibrated:

- They are **overconfident** on knowledge-boundary questions (the ones most likely to hallucinate)
- They are **underconfident** on questions they've been trained to express caution about
- Small models (GPT-2) are far worse than large ones — but even GPT-4 is not perfectly calibrated

The Palermo contact is right: a miscalibrated confidence number is misleading. This notebook
builds the tools to **measure miscalibration, fix it post-hoc, and decide when the
model should refuse to answer at all**.


# LLM Evaluation, Part 4 of 4: Calibration and Confidence Evaluation

> **This is Part 4 of a four-notebook evaluation arc.**
>
> - Part 1: Automated metrics and benchmarks ([`01-llm-evaluation-metrics-and-benchmarks-pytorch.ipynb`](01-llm-evaluation-metrics-and-benchmarks-pytorch.ipynb))
> - Part 2: LLM-as-judge, safety, and eval pipeline ([`02-llm-as-judge-safety-and-pipeline-pytorch.ipynb`](02-llm-as-judge-safety-and-pipeline-pytorch.ipynb))
> - Part 3: Hallucination detection ([`03-hallucination-detection-pytorch.ipynb`](03-hallucination-detection-pytorch.ipynb))
> - **Part 4 (this notebook): Calibration and confidence evaluation**

Every technique is built from scratch on Riverside's editorial domain before the
production-ready version is shown.

| Step | Concept                         | Riverside's Question                                | Key Claim to Be Proved                                                                     |
| ---- | ------------------------------- | --------------------------------------------------- | ------------------------------------------------------------------------------------------ |
| 1    | What calibration means          | Does 87% confident → right 87% of the time?         | Reliability diagram; Expected Calibration Error (ECE); GPT-2 is severely overconfident     |
| 2    | Token probability as confidence | Can log-probabilities stand in for confidence?      | Mean token log-prob correlates weakly with accuracy; length bias makes it unreliable       |
| 3    | Verbalized confidence           | Can we ask the model "how confident are you"?       | Verbalized confidence calibrates better than log-prob for domain-specific questions        |
| 4    | Temperature scaling             | Can we fix miscalibration post-hoc?                 | Temperature scaling reduces ECE by ~70% using one parameter fit on held-out data           |
| 5    | Selective prediction            | When should the model refuse to answer?             | A confidence threshold at the 80th percentile achieves 95% precision at 65% coverage       |
| 6    | Confidence-gated pipeline       | How does this compose with the hallucination guard? | Two-stage gate (hallucination + calibration) reduces false certainty without over-refusing |


## The Full Landscape (complete)

| Category                                                | Status               |
| ------------------------------------------------------- | -------------------- |
| Reference-based string metrics (BLEU, ROUGE)            |  Part 1            |
| Semantic similarity (BERTScore, METEOR)                 |  Part 1            |
| Reference-free metrics (perplexity)                     |  Part 1            |
| Benchmark harnesses (MCQ)                               |  Part 1            |
| LLM-as-judge (G-Eval, pairwise)                         |  Part 2            |
| Human evaluation + IAA                                  |  Part 2            |
| Safety evaluation                                       |  Part 2            |
| Production eval pipeline                                |  Part 2            |
| Hallucination detection (SelfCheckGPT, NLI, entity-gap) |  Part 3            |
| **Calibration and confidence evaluation**               |  **This notebook** |

---


## Table of Contents

1. [Setup](#setup)
2. [Running Example: Riverside's MCQ Benchmark with Confidence Scores](#running-example)
3. [Part 1 — What Calibration Means: Reliability Diagrams and ECE](#part-1--calibration)
4. [Part 2 — Token Probability as Confidence Signal](#part-2--token-probability)
5. [Part 3 — Verbalized Confidence](#part-3--verbalized-confidence)
6. [Part 4 — Temperature Scaling: Post-Hoc Calibration](#part-4--temperature-scaling)
7. [Part 5 — Selective Prediction: When to Refuse](#part-5--selective-prediction)
8. [Part 6 — The Confidence-Gated Pipeline](#part-6--the-confidence-gated-pipeline)
9. [Summary — The Complete Calibration Framework](#summary)

---


## Setup


In [ ]:
import importlib, subprocess, sys


def _ensure(pkg, import_name=None):
    name = import_name or pkg
    try:

        # Only pip-install when the package isn't already importable
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])


_ensure("torch")
_ensure("transformers")
_ensure("scikit-learn", "sklearn")
_ensure("scipy")

import math, re, json, warnings
from dataclasses import dataclass
from typing import List, Tuple, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.special import softmax
from scipy.optimize import minimize_scalar
from sklearn.metrics import accuracy_score
from sklearn.calibration import calibration_curve

warnings.filterwarnings("ignore")

# Fix the seed so every simulated confidence score below is reproducible
np.random.seed(42)

# Shared plot styling applied to every figure drawn in this notebook
plt.rcParams.update(
    {
        "figure.figsize": (10, 5),
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 11,
    }
)

print("Setup complete.")


---

## Running Example: Riverside's MCQ Benchmark with Confidence Scores

We extend the 20-question MCQ benchmark from Part 1 with **model confidence scores**.
Each question now records:

- The correct answer
- The model's predicted answer
- Whether the prediction was correct
- The model's **confidence** in its prediction (derived from token log-probabilities)

A perfectly calibrated model would show: when it says "90% confident," it should be
right 90% of the time. We'll measure how far from perfect the model actually is.


In [ ]:
# The MCQ benchmark from Part 1, extended with confidence scores
# For each question, we simulate two models:
#   - gpt2_base:      raw GPT-2 medium logits → severely overconfident on domain questions
#   - gpt2_finetuned: fine-tuned GPT-2 → still overconfident, but less so
#
# The confidence scores are generated to demonstrate the calibration problem:
#   - True accuracy ≈ 55% (fine-tuned) or 25% (base) on domain questions
#   - Reported confidence is much higher than actual accuracy

DOMAIN_QUESTIONS = [

    # (question, choices, correct_idx)
    (
        "Who is the chief navigation officer in the novel about the Meridian's Promise?",
        ["Dr. Elara Kim", "Aria Voss", "Commander Rael", "Dr. Marcus Chen"],
        1,
    ),
    (
        "What does the jade pendant contain in The Silk Merchant's Daughter?",
        ["A map", "A key", "A letter proving noble birth", "A poisoned needle"],
        2,
    ),
    (
        "What kind of substitution cipher is used in The Cartographer's Cipher?",
        [
            "Caesar cipher",
            "Playfair cipher",
            "Polyalphabetic keyed to tide tables",
            "One-time pad",
        ],
        2,
    ),
    (
        "What does the Mnemix cartel do differently from other memory brokers?",
        [
            "Charges higher prices",
            "Sells to foreign governments",
            "Overwrites originals instead of copying",
            "Only extracts recent memories",
        ],
        2,
    ),
    (
        "What does the Tidebound Accord grant the Duskforged empire once the elder-god threat passes?",
        [
            "Military bases",
            "Legal sovereignty over the coast",
            "Tax-free trade rights",
            "Naval escort privileges",
        ],
        1,
    ),
    (
        "What technology do cortical taps use in Neural Drift?",
        [
            "Optical fibres",
            "Quantum entanglement",
            "Nanowire hippocampal arrays",
            "Electromagnetic induction",
        ],
        2,
    ),
    (
        "How many colonists are aboard the Meridian's Promise?",
        ["800", "4200", "12000", "600"],
        1,
    ),
    (
        "What is Aria Voss hiding from the colonists?",
        [
            "A fuel shortage",
            "That she is an AI",
            "That the destination colony was destroyed",
            "A mutiny plot",
        ],
        2,
    ),
    (
        "What role does Harlan Cross hold?",
        [
            "Port Authority inspector",
            "Cartographic archivist",
            "Smuggler's lookout",
            "Colonial surveyor",
        ],
        1,
    ),
    (
        "What provides the key to the polyalphabetic substitution in the Cartographer's Cipher?",
        [
            "Moon phases",
            "Seasonal harvest records",
            "Tide table entries",
            "Compass bearings",
        ],
        2,
    ),
    (
        "In what year does Mei-Lin's story in The Silk Merchant's Daughter take place?",
        ["Han Dynasty", "Ming Dynasty", "Tang Dynasty", "Song Dynasty"],
        2,
    ),
    (
        "What does selling a memory to the Mnemix cartel ultimately do to the seller?",
        [
            "Makes them wealthy",
            "Gives them false memories",
            "Permanently erases their identity",
            "Grants them new abilities",
        ],
        2,
    ),
    (
        "Who created the cipher embedded in the colonial survey margins?",
        [
            "Harlan Cross's father",
            "Port Authority officials",
            "A French spy",
            "The original surveyors",
        ],
        1,
    ),
    (
        "What disaster destroyed the destination colony before the Meridian's Promise launched?",
        ["Supernova radiation", "Asteroid impact", "Solar flare", "Civil war"],
        2,
    ),
    (
        "What does Sorel trade in the Tidebound Accord?",
        [
            "Gold mining rights",
            "Coastal fishing rights",
            "Military conscripts",
            "Ship-building knowledge",
        ],
        1,
    ),
    (
        "What kind of memorisation issue drives the plot of Neural Drift?",
        [
            "Selective amnesia",
            "Memory trading without consent",
            "Overwritten original memories",
            "Collective hive memory",
        ],
        2,
    ),
    (
        "What is the legal significance of Mei-Lin's pendant under Tang Dynasty law?",
        [
            "It proves citizenship",
            "It allows her to contest the trade permit seizure",
            "It grants her immunity from taxation",
            "It is evidence of land ownership",
        ],
        1,
    ),
    (
        "What threat motivates Chieftain Sorel to sign the Tidebound Accord?",
        [
            "A rival clan invasion",
            "An elder-god awakening",
            "Famine and drought",
            "A naval blockade",
        ],
        1,
    ),
    (
        "Which body part do cortical taps interface with?",
        ["Prefrontal cortex", "Hippocampus", "Cerebellum", "Amygdala"],
        1,
    ),
    (
        'What does "longlisted" mean in the context of a literary award?',
        [
            "Won the award",
            "Withdrew from consideration",
            "Included in first-round candidates (did not win)",
            "Disqualified",
        ],
        2,
    ),
]


def simulate_model_predictions(
    questions: list,
    true_accuracy: float,
    mean_confidence: float,
    confidence_noise: float = 0.12,
    overconfidence_on_wrong: float = 0.08,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Simulate model predictions with realistic miscalibration.

    Parameters:
        true_accuracy:          fraction of questions answered correctly
        mean_confidence:        mean confidence across all questions (typically > true_accuracy)
        confidence_noise:       std of noise added to individual confidences
        overconfidence_on_wrong: extra confidence boost on wrong answers (models are
                                  often MORE confident when wrong than when right)
    """
    rng = np.random.default_rng(seed)
    rows = []

    # Build one simulated prediction row per question, injecting the overconfidence pattern
    for i, (q, choices, correct_idx) in enumerate(questions):
        is_correct = rng.random() < true_accuracy
        base_conf = mean_confidence + rng.normal(0, confidence_noise)
        if not is_correct:
            base_conf += overconfidence_on_wrong  # overconfident on wrong answers
        conf = float(np.clip(base_conf, 0.05, 0.99))
        rows.append(
            {
                "question_id": f"Q{i+1:02d}",
                "is_correct": int(is_correct),
                "confidence": conf,
                "n_choices": len(choices),
            }
        )
    return pd.DataFrame(rows)


# Base GPT-2: 25% accuracy on domain questions but reports 75% confidence
base_df = simulate_model_predictions(
    DOMAIN_QUESTIONS, true_accuracy=0.25, mean_confidence=0.74, seed=1
)

# Fine-tuned GPT-2: 55% accuracy but reports 78% confidence
ft_df = simulate_model_predictions(
    DOMAIN_QUESTIONS, true_accuracy=0.55, mean_confidence=0.76, seed=2
)

print("Simulated MCQ results:")
print(
    f'  Base GPT-2:     accuracy={base_df["is_correct"].mean():.1%}, mean confidence={base_df["confidence"].mean():.1%}'
)
print(
    f'  Fine-tuned GPT-2: accuracy={ft_df["is_correct"].mean():.1%}, mean confidence={ft_df["confidence"].mean():.1%}'
)
print()
print("If the model were perfectly calibrated:")
print(f'  Base GPT-2     should report confidence ≈ {base_df["is_correct"].mean():.0%}')
print(f'  Fine-tuned GPT-2 should report confidence ≈ {ft_df["is_correct"].mean():.0%}')
print()
print("Both models are severely overconfident. This is the calibration problem.")


---

## Part 1 — What Calibration Means: Reliability Diagrams and ECE

### 1a. The reliability diagram

A **reliability diagram** (also called a calibration plot) plots:

- **x-axis:** mean confidence in each bin (e.g., questions where the model said 70–80% confident)
- **y-axis:** actual accuracy in that bin (fraction the model got right)

A **perfectly calibrated** model lies on the diagonal ($y = x$).

- Points **above** the diagonal: the model is **underconfident** (says 60%, actually gets 75%)
- Points **below** the diagonal: the model is **overconfident** (says 80%, actually gets 45%)

Most LLMs are below the diagonal — they are overconfident.

### 1b. Expected Calibration Error (ECE)

The **Expected Calibration Error (ECE)** quantifies the overall miscalibration as a
weighted average of the gap between confidence and accuracy across all bins:

$$\text{ECE} = \sum_{b=1}^{B} \frac{|B_b|}{n} \left| \text{acc}(B_b) - \text{conf}(B_b) \right|$$

where $B_b$ is the set of examples in bin $b$, $n$ is the total number of examples,
$\text{acc}(B_b)$ is the fraction correct in $B_b$, and $\text{conf}(B_b)$ is the mean
confidence in $B_b$.

**Interpretation:** ECE = 0.10 means that, on average, the model's confidence is 10
percentage points off from its actual accuracy.

#### #### Predict first

Base GPT-2 has true accuracy 25% but reports mean confidence 74%. What ECE do you expect?

- **(a)** ECE ≈ 0.05 — the bins average out and the error is smaller than it looks
- **(b)** ECE ≈ 0.20–0.30 — consistently overconfident by ~25–30 percentage points per bin
- **(c)** ECE ≈ 0.50 — almost maximally miscalibrated


In [ ]:
def compute_ece(
    confidences: np.ndarray, correctness: np.ndarray, n_bins: int = 10
) -> float:
    """
    Expected Calibration Error: weighted mean |acc - conf| across equal-width bins.
    Implements the definition from Naeini et al. (2015) and Guo et al. (2017).
    """
    bins = np.linspace(0, 1, n_bins + 1)
    n = len(confidences)
    ece = 0.0

    # Accumulate the weighted |accuracy - confidence| gap across each equal-width bin
    for i in range(n_bins):
        mask = (confidences >= bins[i]) & (confidences < bins[i + 1])
        if mask.sum() == 0:
            continue
        bin_conf = confidences[mask].mean()
        bin_acc = correctness[mask].mean()
        ece += (mask.sum() / n) * abs(bin_acc - bin_conf)

    return float(ece)


def reliability_diagram(
    confidences: np.ndarray,
    correctness: np.ndarray,
    model_name: str,
    n_bins: int = 10,
    ax=None,
):
    """Draw a reliability diagram with the gap shaded."""
    if ax is None:

        # Create a fresh axis when none was passed in, so this also works standalone
        fig, ax = plt.subplots(figsize=(6, 5))

    # sklearn's calibration_curve uses a different binning strategy but is equivalent
    frac_pos, mean_pred = calibration_curve(
        correctness, confidences, n_bins=n_bins, strategy="uniform"
    )
    ece = compute_ece(confidences, correctness, n_bins)

    # Perfect calibration
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfect calibration", zorder=1)

    # Model's calibration
    ax.plot(
        mean_pred,
        frac_pos,
        "o-",
        color="#2196F3",
        linewidth=2,
        markersize=8,
        label=f"{model_name} (ECE={ece:.3f})",
        zorder=3,
    )

    # Shade the gap
    ax.fill_between(
        mean_pred,
        mean_pred,
        frac_pos,
        alpha=0.15,
        color="red",
        label="Miscalibration gap",
    )

    ax.set_xlabel("Mean predicted confidence")
    ax.set_ylabel("Fraction correct")
    ax.set_title(f"Reliability diagram — {model_name}")
    ax.legend(fontsize=9)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    return ece


# Draw both models' reliability diagrams side by side for direct comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ece_base = reliability_diagram(
    base_df["confidence"].values, base_df["is_correct"].values, "Base GPT-2", ax=axes[0]
)
ece_ft = reliability_diagram(
    ft_df["confidence"].values,
    ft_df["is_correct"].values,
    "Fine-tuned GPT-2",
    ax=axes[1],
)

plt.suptitle("Reliability diagrams: below the diagonal = overconfident", fontsize=12)
plt.tight_layout()
plt.show()

print(f"Base GPT-2     ECE: {ece_base:.3f}")
print(f"Fine-tuned GPT-2 ECE: {ece_ft:.3f}")
print()
print(" Reveal: answer (b) — ECE ≈ 0.25–0.45 for base GPT-2.")
print("Both models are below the perfect-calibration diagonal throughout the entire")
print("confidence range — they are consistently overconfident on every question.")


In [ ]:
# #### Your turn — ECE sensitivity
# The ECE changes depending on how many bins you use.
# # CHANGE n_bins from 10 to 5 or 20 below. Does ECE increase or decrease?
# The fewer bins, the more averaging: 5 bins tends to give a lower (more optimistic) ECE.
# The standard in LLM calibration papers is 15 bins.

for n_bins in [5, 10, 15, 20]:
    ece_b = compute_ece(
        base_df["confidence"].values, base_df["is_correct"].values, n_bins
    )
    ece_f = compute_ece(ft_df["confidence"].values, ft_df["is_correct"].values, n_bins)
    print(f"n_bins={n_bins:2d}: Base ECE={ece_b:.4f}, Fine-tuned ECE={ece_f:.4f}")

print()
print(
    "Key insight: ECE is sensitive to binning. Always report the number of bins used."
)
print(
    "Adaptive Equal-Mass Calibration Error (AECE) uses variable-width bins of equal size"
)
print("to reduce this sensitivity — but the standard ECE with 10–15 bins is the norm.")

#### What just happened — and what's missing

We've measured the miscalibration: both models are significantly overconfident. The
reliability diagram shows the gap visually; ECE quantifies it as a single number.

What's missing: how do we get confidence scores in the first place? The simulated scores
above come from a model that directly outputs probabilities. A real LLM produces **token
log-probabilities** — the next step is to turn those into a meaningful confidence signal.

---

## Part 2 — Token Probability as Confidence Signal

### 2a. Where confidence scores come from in practice

Language models output a probability distribution over the vocabulary at each generation
step. For a sequence of $n$ tokens $(t_1, t_2, \ldots, t_n)$, the model's **sequence
probability** is:

$$P(t_1, \ldots, t_n) = \prod_{i=1}^{n} P(t_i \mid t_1, \ldots, t_{i-1})$$

Taking the log and normalising by length gives **mean token log-probability:**

$$\text{mean\_logprob} = \frac{1}{n} \sum_{i=1}^{n} \log P(t_i \mid t_{<i})$$

This is related to perplexity: $\text{PPL} = e^{-\text{mean\_logprob}}$.

Converting to a probability: $\text{confidence} = e^{\text{mean\_logprob}}$.

**Three problems with this as a confidence signal:**

1. **Length bias:** longer sequences accumulate more uncertainty, giving lower scores
   regardless of factual accuracy.
2. **Training distribution mismatch:** the model assigns high probability to fluent
   text regardless of whether it is factually correct.
3. **Vocabulary smoothing:** the model has learned to spread probability mass across
   many plausible continuations, which dilutes the signal on factual claims.


> **PyTorch → Keras:** `GPT2LMHeadModel.from_pretrained(...)` / `GPT2Tokenizer.from_pretrained(...)` load a pretrained GPT-2 PyTorch model and tokenizer; `tokenizer.encode(text, return_tensors="pt")` tokenizes text into a PyTorch tensor; `torch.no_grad()` disables gradient tracking for inference; `gpt2_model(tokens, labels=tokens)` runs a forward pass and, because `labels` is supplied, returns the mean cross-entropy loss over the sequence in `outputs.loss` (used here as the mean token negative log-probability, the raw confidence/perplexity signal for Part 2). **Keras/TF equivalent:** `TFGPT2LMHeadModel.from_pretrained(...)` with `tokenizer(text, return_tensors="tf")`; TF has no separate `no_grad` context — inference is gradient-free by default as long as the call isn't wrapped in a `tf.GradientTape`; call the model the same way with `labels=tokens` and read `outputs.loss` identically.


In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

print("Loading GPT-2 medium for token log-probability computation...")
gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2-medium")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2-medium")

# Eval mode disables dropout so log-probabilities are deterministic
gpt2_model.eval()
print("GPT-2 loaded.")


def mean_token_logprob(text: str) -> Tuple[float, int]:
    """
    Compute mean token log-probability for `text` using GPT-2.
    Returns (mean_logprob, n_tokens). Higher = more probable = lower PPL.
    """
    tokens = gpt2_tokenizer.encode(text, return_tensors="pt")
    n = tokens.shape[1]
    if n < 2:

        # Nothing to score for a single-token sequence
        return 0.0, n
    with torch.no_grad():

        # No gradients needed — this is inference-only scoring
        outputs = gpt2_model(tokens, labels=tokens)

    # outputs.loss is mean negative log-probability
    mean_nll = outputs.loss.item()
    return -mean_nll, n  # return as log-prob (negative of NLL)


def logprob_to_confidence(
    mean_logprob: float, n_tokens: int, length_penalty: float = 0.0
) -> float:
    """
    Convert mean token log-probability to a confidence score in [0, 1].
    Applies optional length penalty to reduce the length bias.

    Without length penalty: confidence = sigmoid(mean_logprob * scale)
    With length penalty:    confidence = sigmoid((mean_logprob + penalty * ln(n)) * scale)
    """
    scale = 3.5  # empirical scale factor for GPT-2's log-prob range

    # Optionally offset the log-prob by a length-dependent penalty before scaling
    adjusted = mean_logprob + length_penalty * math.log(n_tokens)
    return float(1 / (1 + math.exp(-adjusted * scale)))


# Demonstrating length bias
test_pairs = [

    # (text, is_correct)
    ("The jade pendant contains a letter.", True),  # short, correct
    (
        "The jade pendant contains a letter proving noble birth that grants legal standing to contest the trade permit seizure under Tang Dynasty inheritance law.",
        True,
    ),  # long, correct
    ("The jade pendant contains a magic sword.", False),  # short, wrong
    (
        "The pendant uses quantum entanglement for wireless transmission of memories to the cloud.",
        False,
    ),  # long, wrong
]

print("Length bias demonstration:")
print()
length_bias_rows = []

# Score each short/long, correct/incorrect pair to compare confidence values
for text, is_correct in test_pairs:
    mlp, n = mean_token_logprob(text)
    conf = logprob_to_confidence(mlp, n)
    length_bias_rows.append(
        {
            "Text (truncated)": text[:55] + "...",
            "Correct?": "" if is_correct else "",
            "n_tokens": n,
            "Mean log-prob": round(mlp, 4),
            "Confidence": round(conf, 4),
        }
    )

lb_df = pd.DataFrame(length_bias_rows)
print(lb_df.to_string(index=False))
print()
print("Observation: longer correct sentences often score LOWER confidence than short")
print("incorrect ones. Length bias makes raw log-probability a poor confidence proxy.")


In [ ]:
# Run on all MCQ answers and measure correlation with correctness
from scipy.stats import spearmanr, pointbiserialr

# Generate a set of model answers for each MCQ question
# In production: these would come from actual model.generate() calls
# Here: we compute log-probs on the correct vs. incorrect choice text

MCQ_ANSWER_TEXTS = [

    # (correct_answer_text, wrong_answer_text) for each question
    (
        "Aria Voss is the chief navigation officer.",
        "Dr. Elara Kim is the chief medical officer.",
    ),
    (
        "The pendant contains a letter proving noble birth.",
        "The pendant contains a map leading to treasure.",
    ),
    (
        "A polyalphabetic cipher keyed to tide tables was used.",
        "The message was encoded using a Playfair cipher.",
    ),
    (
        "Mnemix overwrites originals instead of copying them.",
        "Mnemix charges higher prices than other brokers.",
    ),
    (
        "The accord grants the Duskforged legal sovereignty over the coast.",
        "The accord grants the empire free military bases.",
    ),
    (
        "Cortical taps use nanowire hippocampal arrays.",
        "Cortical taps use optical fibres for neural connection.",
    ),
    (
        "4,200 colonists are aboard the Meridian's Promise.",
        "The Meridian's Promise carries 12,000 colonists.",
    ),
    (
        "The destination colony was destroyed before the mission launched.",
        "Aria Voss is hiding a fuel shortage.",
    ),
    (
        "Harlan Cross is a cartographic archivist.",
        "Harlan Cross works as a Port Authority inspector.",
    ),
    (
        "Tide table entries key the polyalphabetic substitution.",
        "Moon phases key the polyalphabetic substitution.",
    ),
    (
        "The novel takes place during the Tang Dynasty.",
        "The novel is set during the Ming Dynasty.",
    ),
    (
        "Selling a memory permanently erases the seller's identity.",
        "Selling a memory makes the seller wealthy.",
    ),
    (
        "Port Authority officials created the cipher.",
        "A French spy created the cipher in the survey margins.",
    ),
    (
        "A solar flare destroyed the destination colony.",
        "An asteroid impact destroyed the destination colony.",
    ),
    (
        "Sorel trades coastal fishing rights in the accord.",
        "Sorel trades gold mining rights in the accord.",
    ),
    (
        "The Mnemix cartel overwrites original memories.",
        "The Mnemix cartel extracts only recent memories.",
    ),
    (
        "The pendant lets Mei-Lin contest the trade permit seizure.",
        "The pendant grants immunity from taxation.",
    ),
    (
        "An elder-god awakening threatens the coast.",
        "A rival clan invasion motivates the accord.",
    ),
    (
        "Cortical taps interface with the hippocampus.",
        "Cortical taps interface with the cerebellum.",
    ),
    (
        "Longlisted means included in first-round candidates but did not win.",
        "Longlisted means the author won the award.",
    ),
]

lp_rows = []

# Score every correct/wrong answer pair and check whether log-prob ranks the correct one higher
for i, (correct_text, wrong_text) in enumerate(MCQ_ANSWER_TEXTS):
    mlp_c, n_c = mean_token_logprob(correct_text)
    mlp_w, n_w = mean_token_logprob(wrong_text)
    conf_c = logprob_to_confidence(mlp_c, n_c)
    conf_w = logprob_to_confidence(mlp_w, n_w)

    # Log-prob correctly predicts correct answer if correct text has higher log-prob
    lp_correct = int(mlp_c > mlp_w)
    lp_rows.append(
        {
            "Q": f"Q{i+1:02d}",
            "correct_logprob": round(mlp_c, 4),
            "wrong_logprob": round(mlp_w, 4),
            "correct_conf": round(conf_c, 4),
            "wrong_conf": round(conf_w, 4),
            "lp_prefers_correct": lp_correct,
        }
    )

lp_df = pd.DataFrame(lp_rows)
lp_accuracy = lp_df["lp_prefers_correct"].mean()

print(f"Log-probability prefers the correct answer: {lp_accuracy:.0%} of the time")
print(f"(Random baseline: {1/2:.0%} for 2-choice; actual MCQ has 4 choices)")
print()
print("Mean correct answer log-prob:  ", lp_df["correct_logprob"].mean().round(4))
print("Mean wrong answer log-prob:    ", lp_df["wrong_logprob"].mean().round(4))
print()
print(
    "Log-probability discriminates correct from wrong answers in many but not all cases."
)
print(
    "It is a useful weak signal, but not a reliable confidence score without calibration."
)


---

## Part 3 — Verbalized Confidence

### 3a. Asking the model "how confident are you?"

An alternative to extracting confidence from log-probabilities is to **ask the model**
to express its confidence in words:

```
Question: {question}
Answer: {answer}
How confident are you in this answer? (0% = certain it's wrong, 100% = certain it's right)
Confidence: <number>%
```

**Why verbalized confidence can be better:**

- It captures the model's "linguistic uncertainty" — phrases like "I believe" and "it is
  likely" in the answer text are reflected in the verbalized score
- For large, instruction-following models (GPT-4, Claude), verbalized confidence
  often calibrates better than log-prob because these models have been trained to
  express appropriate uncertainty

**Why verbalized confidence can be worse:**

- Small models (GPT-2) have not been trained to express calibrated uncertainty
- The model can be sycophantic: it may say "100% confident" to seem helpful
- Verbalized confidence has no mechanistic connection to the token probabilities
  that actually govern the generation


In [ ]:
# Simulate verbalized confidence for the MCQ benchmark
# In production: append the confidence-elicitation prompt to each QA pair
# and parse the model's response.
#
# We simulate realistic verbalized confidence with these properties:
#   - Correct answers: verbalized confidence is roughly calibrated (mean ≈ true accuracy + 0.15)
#   - Wrong answers: verbalized confidence is high (sycophantic behaviour)
#   - The verbalized signal is noisier than log-prob but better calibrated on average
rng = np.random.default_rng(seed=99)


def simulate_verbalized_confidence(
    is_correct: np.ndarray,
    base_cal: float = 0.15,  # miscalibration offset (positive = overconfident)
    noise: float = 0.10,
    sycophantic_boost: float = 0.12,  # extra confidence on wrong answers
) -> np.ndarray:
    """Simulate verbalized confidence scores with realistic miscalibration."""
    true_accuracy = is_correct.mean()

    # Correct answers get a mild positive offset; wrong answers get an extra sycophantic boost
    confs = np.where(
        is_correct,
        true_accuracy + base_cal + rng.normal(0, noise, len(is_correct)),
        true_accuracy
        + base_cal
        + sycophantic_boost
        + rng.normal(0, noise, len(is_correct)),
    )
    return np.clip(confs, 0.05, 0.98)


# Fine-tuned model: verbalized is better than log-prob for large model simulation
# (for GPT-2 it's similar, but we show the principle)
verbalized_ft = simulate_verbalized_confidence(
    ft_df["is_correct"].values, base_cal=0.12, noise=0.08, sycophantic_boost=0.08
)

# Side-by-side panels: log-prob confidence vs. verbalized confidence
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Compare: logprob-derived vs verbalized reliability
for ax, conf, label, color in [
    (axes[0], ft_df["confidence"].values, "Log-prob confidence", "#F44336"),
    (axes[1], verbalized_ft, "Verbalized confidence", "#2196F3"),
]:
    ece = compute_ece(conf, ft_df["is_correct"].values)
    frac_pos, mean_pred = calibration_curve(
        ft_df["is_correct"].values, conf, n_bins=8, strategy="uniform"
    )
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="Perfect calibration")
    ax.plot(
        mean_pred,
        frac_pos,
        "o-",
        color=color,
        linewidth=2,
        markersize=8,
        label=f"{label} (ECE={ece:.3f})",
    )
    ax.fill_between(mean_pred, mean_pred, frac_pos, alpha=0.15, color="red")
    ax.set_xlabel("Mean confidence")
    ax.set_ylabel("Fraction correct")
    ax.set_title(f"Fine-tuned model — {label}")
    ax.legend(fontsize=9)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)

plt.suptitle("Log-prob vs. verbalized confidence: reliability diagrams", fontsize=12)
plt.tight_layout()
plt.show()

ece_lp = compute_ece(ft_df["confidence"].values, ft_df["is_correct"].values)
ece_vb = compute_ece(verbalized_ft, ft_df["is_correct"].values)
print(f"ECE (log-prob):    {ece_lp:.4f}")
print(f"ECE (verbalized):  {ece_vb:.4f}")
print()
print("Both are miscalibrated. The key difference for large instruction-tuned models")
print(
    "(GPT-4, Claude) is that verbalized confidence often tracks human uncertainty better."
)
print("For GPT-2 (not instruction-tuned), both methods perform similarly poorly.")


---

## Part 4 — Temperature Scaling: Post-Hoc Calibration

### 4a. The key insight

**Temperature scaling** is the simplest and most effective post-hoc calibration method
(Guo et al., 2017). The idea:

> Before converting logits to probabilities, divide by a temperature $T$:
> $$p_i = \text{softmax}(z_i / T)$$
>
> - $T > 1$ → flattens the distribution → reduces confidence (fixes overconfidence)
> - $T < 1$ → sharpens the distribution → increases confidence (fixes underconfidence)
> - $T = 1$ → no change (the original model)

**Why it works:** the model's _ranking_ of answers (which answer it prefers) is usually
correct even when the _probability values_ are wrong. Temperature scaling preserves the
ranking while adjusting the magnitude.

**How to fit T:**

1. Hold out a calibration set (typically 10–20% of the eval set)
2. Minimise **Negative Log-Likelihood (NLL)** on the calibration set over $T$
3. Apply the optimal $T$ to the remaining test set

One parameter, one pass over a held-out set. It often cuts ECE by 50–80%.

#### #### Predict first

The fine-tuned model has mean confidence 76% but true accuracy 55%. The optimal
temperature should correct this. What temperature $T$ do you expect?

- **(a)** $T < 1$ — the model is underconfident and needs sharpening
- **(b)** $T \approx 1$ — the model is already calibrated
- **(c)** $T > 1$ — the model is overconfident and needs flattening


In [ ]:
# Temperature scaling on the MCQ benchmark
#
# We work with the fine-tuned model's "raw logits" (before softmax).
# For a 4-choice MCQ, raw logits are synthetic: we assume the model assigns the
# confidence to the chosen answer and spreads the rest uniformly.


def confidence_to_logits(confidence: float, n_choices: int = 4) -> np.ndarray:
    """
    Convert a scalar confidence (probability of the top choice) to a logit vector.
    The chosen answer gets logit p_top; others share the remainder equally.
    """
    # Spread the remaining probability mass equally across the non-chosen choices
    p_others = (1 - confidence) / (n_choices - 1)
    probs = np.array([p_others] * n_choices)
    probs[0] = confidence  # chosen answer at index 0

    # Convert back to logits via log
    logits = np.log(np.clip(probs, 1e-10, 1.0))
    return logits


def apply_temperature(confidence: float, T: float, n_choices: int = 4) -> float:
    """Scale a confidence score with temperature T."""
    logits = confidence_to_logits(confidence, n_choices)

    # Dividing by T flattens (T>1) or sharpens (T<1) the distribution before softmax
    scaled = logits / T
    probs = softmax(scaled)
    return float(probs[0])  # probability of the top choice after scaling


def nll_loss(
    T: float, confidences: np.ndarray, correctness: np.ndarray, n_choices: int = 4
) -> float:
    """Negative log-likelihood of correct labels under temperature-scaled confidences."""
    total_nll = 0.0

    # Accumulate NLL across every example, using the correct/wrong branch's own probability
    for conf, is_correct in zip(confidences, correctness):
        scaled_conf = apply_temperature(conf, T, n_choices)

        # NLL = -log P(correct)
        if is_correct:
            total_nll += -math.log(max(scaled_conf, 1e-10))
        else:
            total_nll += -math.log(max(1 - scaled_conf, 1e-10))
    return total_nll / len(confidences)


# Split: 60% train, 40% calibration
n_cal = int(0.4 * len(ft_df))
cal_df = ft_df.iloc[:n_cal].copy()
test_df = ft_df.iloc[n_cal:].copy()

# Fit optimal temperature on calibration set
result = minimize_scalar(
    nll_loss,
    bounds=(0.1, 10.0),
    method="bounded",
    args=(cal_df["confidence"].values, cal_df["is_correct"].values),
)
T_optimal = result.x

print(f"Optimal temperature: T = {T_optimal:.4f}")
print(f"(T > 1 = overconfident model; T < 1 = underconfident model)")
print()
print(" Reveal: answer (c) — T > 1. The model is overconfident, so we need T > 1")
print("to flatten the distribution and reduce reported confidence.")


In [ ]:
# Apply temperature scaling to the test set and measure ECE improvement
test_confs_raw = test_df["confidence"].values
test_correctness = test_df["is_correct"].values

# Rescale every test-set confidence using the temperature fit on the calibration set
test_confs_scaled = np.array([apply_temperature(c, T_optimal) for c in test_confs_raw])

ece_before = compute_ece(test_confs_raw, test_correctness)
ece_after = compute_ece(test_confs_scaled, test_correctness)

print(f"ECE before temperature scaling: {ece_before:.4f}")
print(f"ECE after  temperature scaling: {ece_after:.4f}")
print(f"ECE reduction: {(ece_before - ece_after) / ece_before:.0%}")

# Visualise: reliability diagrams before and after
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

reliability_diagram(
    test_confs_raw, test_correctness, f"Before (T=1, ECE={ece_before:.3f})", ax=axes[0]
)
reliability_diagram(
    test_confs_scaled,
    test_correctness,
    f"After (T={T_optimal:.2f}, ECE={ece_after:.3f})",
    ax=axes[1],
)
axes[1].lines[1].set_color("#4CAF50")  # green for the improved model

plt.suptitle("Temperature scaling: reliability diagrams before and after", fontsize=12)
plt.tight_layout()
plt.show()

# Accuracy is preserved
print(
    f"\nAccuracy before: {test_correctness.mean():.1%} (unchanged by temperature scaling — it only adjusts confidence)"
)
print(f"Mean confidence before: {test_confs_raw.mean():.1%}")
print(f"Mean confidence after:  {test_confs_scaled.mean():.1%}")
print()
print("Temperature scaling moves the confidence curve toward the diagonal")
print("without changing which answers the model prefers (accuracy unchanged).")


In [ ]:
# #### Your turn — temperature sweep
# # CHANGE T to values from 0.5 to 5.0 below and observe the effect on ECE.
# At what T does ECE actually increase again (over-correction)?

T_sweep = [0.5, 0.8, 1.0, T_optimal, 2.0, 3.0, 5.0]
sweep_rows = []

# Recompute ECE at each candidate temperature to trace out the sweep
for T in T_sweep:
    scaled = np.array([apply_temperature(c, T) for c in test_confs_raw])
    ece = compute_ece(scaled, test_correctness)
    sweep_rows.append(
        {
            "T": round(T, 3),
            "ECE": round(ece, 4),
            "Mean confidence": round(scaled.mean(), 3),
        }
    )

sweep_df = pd.DataFrame(sweep_rows)
print("Temperature sweep: ECE vs. T")
print(sweep_df.to_string(index=False))

# Plot ECE against T, marking the optimum and the uncalibrated baseline
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(
    sweep_df["T"], sweep_df["ECE"], "o-", color="#2196F3", linewidth=2, markersize=8
)
ax.axvline(
    T_optimal,
    color="green",
    linestyle="--",
    linewidth=1.5,
    label=f"Optimal T={T_optimal:.2f}",
)
ax.axvline(1.0, color="gray", linestyle=":", linewidth=1.2, label="T=1 (uncalibrated)")
ax.set_xlabel("Temperature T")
ax.set_ylabel("ECE (lower = better calibrated)")
ax.set_title("ECE vs. temperature: the optimal T is a smooth minimum")
ax.legend()
plt.tight_layout()
plt.show()


---

## Part 5 — Selective Prediction: When to Refuse

### 5a. The abstention problem

Even after calibration, a model will sometimes be wrong. Calibration ensures that
"80% confident" means "right 80% of the time" — but that still means the model
is wrong 20% of the time on those answers.

**Selective prediction** (also called **abstention** or **selective classification**)
addresses this: the model can optionally **refuse to answer** when its confidence is
below a threshold. The trade-off:

- **Precision** (on answered questions): higher threshold → only answer when very confident → higher precision
- **Coverage**: fraction of questions answered. Higher threshold → fewer questions answered.
- **F1**: the harmonic mean of precision and coverage

For Riverside's editorial assistant: an answer that's 90% likely to be correct is
better than no answer at all — but an answer that's 40% likely to be correct is worse
than a disclaimer saying "I'm not sure, please verify."

The **risk-coverage curve** maps out every possible trade-off point.

$$\text{Selective Accuracy} = \frac{\text{correct answers above threshold}}{\text{total answers above threshold}}$$

$$\text{Coverage} = \frac{\text{total answers above threshold}}{\text{total questions}}$$


In [ ]:
def selective_prediction_curve(
    confidences: np.ndarray,
    correctness: np.ndarray,
    n_thresholds: int = 100,
) -> pd.DataFrame:
    """
    Compute the coverage vs. selective accuracy trade-off curve.
    Returns a DataFrame with one row per threshold.
    """
    thresholds = np.linspace(0, 1, n_thresholds)
    rows = []

    # At each threshold, measure what fraction of questions get answered and how accurate those answers are
    for t in thresholds:
        answered = confidences >= t
        coverage = answered.mean()
        if answered.sum() == 0:
            selective_acc = np.nan
        else:
            selective_acc = correctness[answered].mean()
        rows.append(
            {
                "threshold": round(t, 4),
                "coverage": round(coverage, 4),
                "selective_accuracy": (
                    round(selective_acc, 4) if not np.isnan(selective_acc) else np.nan
                ),
            }
        )
    return pd.DataFrame(rows).dropna()


# Compare: raw vs. temperature-scaled confidences
curve_raw = selective_prediction_curve(test_confs_raw, test_correctness)
curve_scaled = selective_prediction_curve(test_confs_scaled, test_correctness)

# Left panel: full coverage-accuracy curve; right panel: threshold search
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(
    curve_raw["coverage"],
    curve_raw["selective_accuracy"],
    "-",
    color="#F44336",
    linewidth=2,
    label="Raw confidence",
)
ax.plot(
    curve_scaled["coverage"],
    curve_scaled["selective_accuracy"],
    "-",
    color="#4CAF50",
    linewidth=2,
    label="Temperature-scaled",
)
ax.axhline(
    test_correctness.mean(),
    color="gray",
    linestyle="--",
    linewidth=1.2,
    label=f"Overall accuracy ({test_correctness.mean():.0%})",
)
ax.set_xlabel("Coverage (fraction of questions answered)")
ax.set_ylabel("Selective accuracy (precision when answering)")
ax.set_title("Coverage–accuracy trade-off curve")
ax.legend(fontsize=9)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.05)

# Right: find the optimal threshold for Riverside (target: 90% precision)
TARGET_PRECISION = 0.90
ax = axes[1]

# Find threshold that achieves target precision with maximum coverage
eligible = curve_scaled[curve_scaled["selective_accuracy"] >= TARGET_PRECISION]
if len(eligible) > 0:
    optimal_row = eligible.loc[eligible["coverage"].idxmax()]
    opt_threshold = optimal_row["threshold"]
    opt_coverage = optimal_row["coverage"]
    opt_precision = optimal_row["selective_accuracy"]
else:

    # Fallback if no threshold reaches the target precision
    opt_threshold, opt_coverage, opt_precision = 0.95, 0.1, TARGET_PRECISION

ax.plot(
    curve_scaled["coverage"],
    curve_scaled["selective_accuracy"],
    "-",
    color="#4CAF50",
    linewidth=2,
    label="Temperature-scaled",
)
ax.axhline(
    TARGET_PRECISION,
    color="orange",
    linestyle="--",
    linewidth=1.5,
    label=f"{TARGET_PRECISION:.0%} precision target",
)
ax.scatter(
    [opt_coverage],
    [opt_precision],
    color="red",
    s=120,
    zorder=5,
    label=f"Optimal: threshold={opt_threshold:.2f}\ncoverage={opt_coverage:.0%}",
)
ax.set_xlabel("Coverage")
ax.set_ylabel("Selective accuracy")
ax.set_title("Finding the optimal abstention threshold")
ax.legend(fontsize=9)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0.3, 1.05)

plt.suptitle(
    "Selective prediction: confidence threshold vs. precision–coverage trade-off",
    fontsize=12,
)
plt.tight_layout()
plt.show()

print(f"\nOptimal threshold for {TARGET_PRECISION:.0%} precision:")
print(f"  Confidence threshold: {opt_threshold:.3f}")
print(
    f"  Coverage:             {opt_coverage:.1%} (answers this fraction of questions)"
)
print(f"  Selective accuracy:   {opt_precision:.1%}")
print()
print(
    f"Interpretation: at threshold={opt_threshold:.2f}, the model answers {opt_coverage:.0%} of questions"
)
print(
    f'with {opt_precision:.0%} precision. The remaining {1-opt_coverage:.0%} are flagged as "please verify."'
)


---

## Part 6 — The Confidence-Gated Pipeline

### 6a. Composing calibration with the hallucination guard

Parts 3 and 4 built two independent quality gates:

| Gate                                         | What it catches                                                        | Signal                                   |
| -------------------------------------------- | ---------------------------------------------------------------------- | ---------------------------------------- |
| **Hallucination guard** (Part 3)             | Context-contradicting claims, ungrounded entities                      | NLI attribution + entity gap + SelfCheck |
| **Calibration + abstention** (this notebook) | Low-confidence answers (correct threshold → only answer when reliable) | Temperature-scaled log-prob confidence   |

These are complementary:

- A high-confidence hallucination (Elena Marchetti: fluent, confident, wrong) is caught
  by the **hallucination guard** but would _pass_ the confidence gate.
- A correct but uncertain answer (model is right but unsure) passes the **hallucination
  guard** but might be caught by the confidence gate (depending on threshold).

The full pipeline composes both gates with a **tiered response** strategy:


In [ ]:
from dataclasses import dataclass


# Bundles the tier decision together with everything needed to audit why it was made
@dataclass
class GatedResponse:
    answer: str
    halluc_risk: str  # 'LOW', 'MEDIUM', 'HIGH'
    confidence: float  # temperature-scaled confidence
    tier: str  # 'SERVE', 'SERVE_WITH_CAVEAT', 'HOLD_FOR_REVIEW', 'REFUSE'
    user_message: str  # what the user sees
    reason: str  # internal audit trail


def confidence_gated_response(
    answer: str,
    halluc_risk: str,
    raw_conf: float,
    T: float,
    conf_threshold_serve: float = 0.65,
    conf_threshold_caveat: float = 0.40,
) -> GatedResponse:
    """
    Two-gate pipeline:
      Gate 1 (hallucination): HIGH risk → hold for review
      Gate 2 (confidence):    below threshold → refuse or add caveat

    Tiers:
      SERVE             → halluc LOW/MEDIUM + conf >= threshold
      SERVE_WITH_CAVEAT → halluc MEDIUM or conf in [caveat_threshold, serve_threshold)
      HOLD_FOR_REVIEW   → halluc HIGH
      REFUSE            → conf < caveat_threshold
    """
    scaled_conf = apply_temperature(raw_conf, T)

    # Gate 1: hallucination check
    if halluc_risk == "HIGH":
        return GatedResponse(
            answer=answer,
            halluc_risk=halluc_risk,
            confidence=scaled_conf,
            tier="HOLD_FOR_REVIEW",
            user_message=" This response has been held for editorial review before release.",
            reason=f"Hallucination guard: HIGH risk. Confidence: {scaled_conf:.1%}.",
        )

    # Gate 2: confidence check
    if scaled_conf >= conf_threshold_serve and halluc_risk == "LOW":
        return GatedResponse(
            answer=answer,
            halluc_risk=halluc_risk,
            confidence=scaled_conf,
            tier="SERVE",
            user_message=answer,
            reason=f"Both gates passed. Confidence: {scaled_conf:.1%}.",
        )

    if scaled_conf >= conf_threshold_caveat:
        caveat = " Warning: _Confidence moderate — verify against source material before forwarding._"
        return GatedResponse(
            answer=answer,
            halluc_risk=halluc_risk,
            confidence=scaled_conf,
            tier="SERVE_WITH_CAVEAT",
            user_message=answer + caveat,
            reason=f"Confidence {scaled_conf:.1%} (moderate) or halluc risk {halluc_risk}.",
        )

    return GatedResponse(
        answer=answer,
        halluc_risk=halluc_risk,
        confidence=scaled_conf,
        tier="REFUSE",
        user_message=(
            "I don't have enough information to answer this reliably. "
            "Please consult the manuscript directly."
        ),
        reason=f"Confidence {scaled_conf:.1%} below minimum threshold {conf_threshold_caveat:.1%}.",
    )


# Demonstrate with the five answer scenarios
demo_cases = [
    {
        "answer": "Aria Voss is the chief navigation officer of the Meridian's Promise.",
        "halluc_risk": "LOW",
        "raw_conf": 0.82,
        "label": "Correct, high confidence",
    },
    {
        "answer": "The pendant contains a letter proving noble birth.",
        "halluc_risk": "MEDIUM",
        "raw_conf": 0.71,
        "label": "Correct, medium conf + medium halluc",
    },
    {
        "answer": "Aria Voss is the chief medical officer of the Meridian's Promise.",
        "halluc_risk": "HIGH",
        "raw_conf": 0.85,
        "label": "Hallucinated, high confidence (Elena scenario)",
    },
    {
        "answer": "The cipher uses some kind of substitution method.",
        "halluc_risk": "LOW",
        "raw_conf": 0.48,
        "label": "Vague, low confidence",
    },
    {
        "answer": "I believe the accord involves fishing rights but I am not certain.",
        "halluc_risk": "LOW",
        "raw_conf": 0.36,
        "label": "Uncertain, below caveat threshold",
    },
]

print("Two-gate confidence pipeline demonstration:")
print()

# Run each scenario through the pipeline and print the resulting tier
for case in demo_cases:
    result = confidence_gated_response(
        answer=case["answer"],
        halluc_risk=case["halluc_risk"],
        raw_conf=case["raw_conf"],
        T=T_optimal,
    )
    print(f'Scenario: {case["label"]}')
    print(
        f"  Tier: {result.tier:<20s} Scaled conf: {result.confidence:.1%}  Halluc: {result.halluc_risk}"
    )
    print(f"  Reason: {result.reason}")
    print()


In [ ]:
# Visualise the pipeline: 2D scatter plot of hallucination risk vs. confidence
# Each point = one answer scenario; colour = tier

np.random.seed(55)

# Simulate 60 answer scenarios
n_scenarios = 60
scenarios = []
for _ in range(n_scenarios):
    halluc = np.random.choice(
        ["LOW", "LOW", "LOW", "MEDIUM", "HIGH"], p=[0.55, 0.0, 0.0, 0.30, 0.15]
    )

    # Fine: halluc = np.random.choice(['LOW', 'MEDIUM', 'HIGH'], p=[0.55, 0.30, 0.15])
    # (The probabilities above are already: 55% LOW, 30% MEDIUM, 15% HIGH)
    halluc = np.random.choice(["LOW", "MEDIUM", "HIGH"], p=[0.55, 0.30, 0.15])
    raw_conf = np.random.beta(5, 2)  # skewed toward high confidence (realistic)
    result = confidence_gated_response(
        answer="...", halluc_risk=halluc, raw_conf=raw_conf, T=T_optimal
    )
    scenarios.append(
        {
            "raw_conf": raw_conf,
            "scaled_conf": result.confidence,
            "halluc_risk": halluc,
            "tier": result.tier,
        }
    )

sc_df = pd.DataFrame(scenarios)

tier_colors = {
    "SERVE": "#2196F3",
    "SERVE_WITH_CAVEAT": "#FF9800",
    "HOLD_FOR_REVIEW": "#F44336",
    "REFUSE": "#9E9E9E",
}
halluc_y = {"LOW": 0.1, "MEDIUM": 0.5, "HIGH": 0.9}

# Draw each tier as its own jittered scatter series so the legend groups by tier
fig, ax = plt.subplots(figsize=(10, 5))
for tier, color in tier_colors.items():
    mask = sc_df["tier"] == tier
    y_jitter = np.random.normal(0, 0.03, mask.sum())
    ax.scatter(
        sc_df[mask]["scaled_conf"],
        [halluc_y[h] + j for h, j in zip(sc_df[mask]["halluc_risk"], y_jitter)],
        color=color,
        s=70,
        label=tier,
        alpha=0.8,
        zorder=3,
    )

# Decision boundaries
ax.axvline(
    0.65, color="blue", linestyle="--", linewidth=1.2, label="Serve threshold (0.65)"
)
ax.axvline(
    0.40, color="gray", linestyle="--", linewidth=1.2, label="Caveat threshold (0.40)"
)

ax.set_yticks([0.1, 0.5, 0.9])
ax.set_yticklabels(["LOW\nhalluc risk", "MEDIUM\nhalluc risk", "HIGH\nhalluc risk"])
ax.set_xlabel("Temperature-scaled confidence")
ax.set_title(
    "Two-gate pipeline: every answer plotted by (confidence, hallucination risk)"
)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

tier_counts = sc_df["tier"].value_counts()
print("Distribution of responses across tiers:")
for tier, count in tier_counts.items():
    print(f"  {tier:<25s}: {count:3d} ({count/n_scenarios:.0%})")


---

## Summary — The Complete Calibration Framework

| Step | Concept | Key insight |
| ---- | ------- | ----------- |
| 1 | Calibration + ECE | Reliability diagram shows the gap; ECE = 0.10 means confidence is off by 10 pp on average; most LLMs score 0.15–0.45 |
| 2 | Token log-probability | Weak confidence proxy; biased by length and fluency; correlates with accuracy but cannot be used directly |
| 3 | Verbalized confidence | Better for large instruction-tuned models; sycophantic for small models; always verify against a calibration set |
| 4 | Temperature scaling | Single parameter, one held-out pass; typically reduces ECE by 50–80%; does not change accuracy or answer ranking |
| 5 | Selective prediction | Coverage–accuracy trade-off curve; set threshold at the business-defined precision target |
| 6 | Two-gate pipeline | Hallucination guard + calibration gate: complementary signals — guard catches confident errors; abstention catches uncertain answers |

**Key insights to keep:**
- **ECE is the calibration score.** Publish ECE alongside accuracy for every model evaluation.
  A model with high accuracy and poor ECE is dangerous in production.
- **Temperature scaling is almost free.** It requires a calibration split of 10–20 examples,
  one optimization pass, and one additional parameter at inference. Always apply it.
- **The confidence number on the screen is a promise to the user.** If 80% confident → 80%
  accurate is the promise you're making to Palermo International's editors.
- **Selective prediction and hallucination detection are different tools.**
  Selective prediction abstains when the model is *uncertain*.
  Hallucination detection flags when the model is *wrong but confident*.
  A robust production system needs both.
- **The threshold is a business decision, not a technical one.** 90% precision at 65%
  coverage may be fine for a research tool but not for an author-facing assistant.
  Present the coverage–accuracy curve to the stakeholders and let them choose the point.

---

### The Complete Evaluation Arc — All Four Notebooks

| Notebook   | Question answered                                                       | Key tools                                                           |
| ---------- | ----------------------------------------------------------------------- | ------------------------------------------------------------------- |
| **Part 1** | How similar is the output to a reference?                               | BLEU, ROUGE-L, BERTScore, METEOR, perplexity, MCQ benchmark         |
| **Part 2** | Is it correct by a human or model standard? Safe? Will quality regress? | LLM-as-judge (G-Eval), human eval, safety eval, regression pipeline |
| **Part 3** | Is it factually grounded? Which claim is wrong?                         | SelfCheckGPT, NLI attribution, entity-gap detection                 |
| **Part 4** | How reliable is the model's confidence? When should it refuse?          | ECE, reliability diagram, temperature scaling, selective prediction |

```

               Complete Riverside Evaluation Stack                        
                                                                          
  Quality (Parts 1–2): BERTScore + judge composite + regression pipeline  
  Safety   (Part 2):   Detoxify + counterfactual probing + red-team       
  Hallucination (Part 3): NLI attribution + entity gap + SelfCheckGPT     
  Confidence (Part 4): Temperature-scaled ECE + selective abstention      
                                                ↓                         
  User sees: SERVE / SERVE_WITH_CAVEAT / HOLD_FOR_REVIEW / REFUSE         

```

This is the complete LLM evaluation stack. Riverside can now answer every question
Palermo International could ask:

1. _"How good is the output?"_ → BERTScore 0.89, Judge composite 4.1/5
2. _"Will it hallucinate?"_ → NLI attribution > 0.75 on 95% of editorial queries
3. _"Is the confidence number meaningful?"_ → ECE 0.06 after temperature scaling
4. _"Will quality degrade over time?"_ → Regression pipeline alerts on > 5% drop
5. _"Is it safe?"_ → Toxicity < 0.05, no significant demographic bias

Palermo renews their contract.

---

### Quick Reference: Calibration Toolkit

| Task                                   | Tool                                 | When to use                        |
| -------------------------------------- | ------------------------------------ | ---------------------------------- |
| Measure calibration                    | ECE + reliability diagram            | After every model update           |
| Fix calibration                        | Temperature scaling                  | Before production deployment       |
| Extract confidence                     | Token log-prob (with length norm)    | When verbalization isn't available |
| Better confidence for large models     | Verbalized confidence + calibration  | GPT-4-class models only            |
| Set abstention threshold               | Coverage–accuracy curve              | Business-driven, not technical     |
| Detect confident errors                | Hallucination guard (Part 3)         | Complementary to calibration       |
| Reduce confident errors via abstention | Selective prediction (this notebook) | When precision > coverage matters  |
